# Creation of Gold table

## NOTE: due to limitations in Databricks Trial, we cannot save this to a table - we'll save it to a file instead.

This script generates a gold table, containing data in a format optimized for Machine Learning.

In a real scenario, we would apply filters by quality, as well as imputation and scaling. For this case, we just use a filter by variance>0.01 across all datasets.

In [0]:
from pyspark.sql.functions import col

# Step 1: Filter probes with variance > 0.01 and present in both datasets
stats_df = spark.table("silver.methylation.methylation_beta_stats")
filtered_probes_df = stats_df.filter(
    (col("dataset_count") == 2) & (col("beta_variance") > 0.01)
).select("probe_id")

# Step 2: Filter and join with long-format beta values
beta_df = spark.table("silver.methylation.methylation_beta")

filtered_df = beta_df.join(filtered_probes_df, on="probe_id", how="inner") \
    .select("sample_id", "probe_id", "beta")

# Step 3: Pivot to wide format (sample_id = row, probe_id = column)
pivot_df = filtered_df.groupBy("sample_id").pivot("probe_id").agg({"beta": "first"})

# Step 4: Save to Parquet file in the gold volume
output_path = "dbfs:/Volumes/gold/methylation/methylation_features/methylation_beta_001.parquet"

pivot_df.write.mode("overwrite").parquet(output_path)

print(f"✅ File saved: {output_path}")


In [0]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Load the wide methylation matrix
df = pd.read_parquet("/dbfs/Volumes/gold/methylation/methylation_features/methylation_beta_001.parquet")
df.set_index("sample_id", inplace=True)

# Step 2: Get sample group labels from Spark
sample_meta = spark.table("silver.methylation.methylation_beta") \
    .select("sample_id", "sample_group") \
    .distinct().toPandas().set_index("sample_id")

# Align sample group labels to data
df["sample_group"] = sample_meta.loc[df.index, "sample_group"].values

# Step 3: Standardize features and run PCA
X = StandardScaler().fit_transform(df.drop(columns="sample_group"))
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Step 4: Plot
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=df.index)
pca_df["sample_group"] = df["sample_group"]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="sample_group", palette="Set2", s=50)
plt.title("PCA of Methylation Data (variance > 0.01)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.grid(True)
plt.legend(title="Sample Group")
plt.tight_layout()
plt.show()


## Save Samples metadata

In [0]:
# Step 0: Load sample_group metadata from Delta table
sample_meta = spark.table("silver.methylation.methylation_beta") \
    .select("sample_id", "sample_group") \
    .distinct().toPandas().set_index("sample_id")

# Save to gold volume as Parquet
sample_meta.to_parquet("/Volumes/gold/methylation/methylation_features/sample_metadata.parquet")

print("✅ Saved sample_group metadata to: gold/methylation/methylation_features/sample_metadata.parquet")
